# NLP Laboratory Assignment 01: Bigrams and Sentence Probability

**Student:** Affan Khan Samid Khan  
**Roll No.:** 421135  
**PRN:** 22310588  
**Faculty:** Dr. Kapil N. Vhatkar

This notebook implements a bigram model on a self-created 10-sentence English corpus. It calculates observed bigram frequencies, maximum-likelihood conditional probabilities, the probabilities of three test sentences, and add-one/Laplace-smoothed comparisons.

## Theory and Method

For adjacent tokens, the MLE is P(current | previous) = count(previous, current) / count(previous). Sentence probability is the product of all adjacent conditional probabilities, including `<s>` and `</s>` boundary transitions. Laplace smoothing uses (count + 1) / (context count + V), where V is the corpus word types plus `</s>`.

In [ ]:
from collections import Counter
from math import prod
import re

corpus = [
    "I love natural language processing.",
    "Natural language processing is interesting.",
    "I love machine learning.",
    "Machine learning helps computers learn.",
    "Computers process natural language.",
    "I study language processing.",
    "Language models predict the next word.",
    "Bigram models use word pairs.",
    "Natural language models learn patterns.",
    "I study machine learning and language models.",
]
test_sentences = [
    "I love natural language processing.",
    "Natural language models learn patterns.",
    "I study machine learning.",
]
token_pattern = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def tokenize(text):
    return token_pattern.findall(text.lower())

def prepare(sentence):
    return ["<s>"] + tokenize(sentence) + ["</s>"]

prepared = [prepare(sentence) for sentence in corpus]
bigram_counts = Counter()
context_counts = Counter()
for tokens in prepared:
    for previous, current in zip(tokens, tokens[1:]):
        bigram_counts[(previous, current)] += 1
        context_counts[previous] += 1

word_types = {word for tokens in prepared for word in tokens
              if word not in {"<s>", "</s>"}}
V = len(word_types) + 1  # all word types plus </s>; <s> is context-only

def p_mle(previous, current):
    denominator = context_counts[previous]
    return bigram_counts[(previous, current)] / denominator if denominator else 0.0

def p_laplace(previous, current):
    return (bigram_counts[(previous, current)] + 1) / (context_counts[previous] + V)

print("Student: Affan Khan Samid Khan | Roll No.: 421135 | PRN: 22310588")
print(f"Corpus sentences: {len(corpus)} | Word types: {len(word_types)} | V: {V}")
print(f"Distinct observed bigrams: {len(bigram_counts)}")
print("\nPREPROCESSED CORPUS")
for index, tokens in enumerate(prepared, 1):
    print(f"{index:02d}. {' '.join(tokens)}")
print("\nTOP 15 BIGRAM FREQUENCIES")
for index, ((previous, current), count) in enumerate(bigram_counts.most_common(15), 1):
    print(f"{index:02d}. ({previous}, {current}) = {count}")
print("\nTOP 15 CONDITIONAL PROBABILITIES")
for (previous, current), count in bigram_counts.most_common(15):
    print(f"P({current}|{previous}) = {count}/{context_counts[previous]} = {p_mle(previous, current):.6f}")

print("\nSENTENCE PROBABILITIES")
for index, sentence in enumerate(test_sentences, 1):
    tokens = prepare(sentence)
    pairs = list(zip(tokens, tokens[1:]))
    mle_factors = [p_mle(a, b) for a, b in pairs]
    laplace_factors = [p_laplace(a, b) for a, b in pairs]
    print(f"Test {index}: {sentence}")
    print("Bigrams:", " | ".join(f"({a}, {b})" for a, b in pairs))
    print("MLE factors:", " x ".join(f"{p:.6g}" for p in mle_factors))
    print(f"Unsmoothed probability: {prod(mle_factors):.12g}")
    print("Laplace factors:", " x ".join(f"{p:.6g}" for p in laplace_factors))
    print(f"Laplace-smoothed probability: {prod(laplace_factors):.12g}\n")

previous, current = "we", "code"
print("ZERO-FREQUENCY DEMONSTRATION")
print(f"({previous}, {current}): count={bigram_counts[(previous, current)]}, "
      f"MLE={p_mle(previous, current):.6f}, Laplace={p_laplace(previous, current):.6f}")
